# 01. Train experiments

이 노트북은 학습만 담당합니다. 아래 설정을 바꾼 뒤 실행하면 결과와 checkpoint가 experiment/seed별 파일로 저장됩니다. 그래프와 표는 `02_analyze_experiments.ipynb`에서 생성합니다.

In [18]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (Path('D:/gt-super') / 'data').exists():
    ROOT = Path('D:/gt-super')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Python 파일을 수정한 뒤에도 커널의 이전 클래스가 남지 않도록 의존성 순서대로 reload합니다.
import importlib
import gt_aux.config as config_module
import gt_aux.data as data_module
import gt_aux.model as model_module
import gt_aux.eval as eval_module
import gt_aux.train as train_module

config_module = importlib.reload(config_module)
data_module = importlib.reload(data_module)
model_module = importlib.reload(model_module)
eval_module = importlib.reload(eval_module)
train_module = importlib.reload(train_module)

ExperimentConfig = config_module.ExperimentConfig
prepare_data = data_module.prepare_data
release_model = train_module.release_model
train_one_experiment = train_module.train_one_experiment

## 학습 설정

여러 seed 실험은 `SEED`만 바꾸어 노트북을 다시 실행합니다. 같은 seed 집합을 모든 experiment에 사용해야 공정하게 비교할 수 있습니다.

In [19]:
RUN_MODE = 'smoke'
SEED = 44
DATA_SEED = 42  # 모든 training seed에서 고정: 동일 train/val split 보장
EXPERIMENTS = ['baseline', 'shared_detach', 'shared_e2e']

TRAIN_IMAGES = 400
VAL_IMAGES = 100
EPOCHS = 7
BATCH_SIZE = 2
NUM_WORKERS = 0

IMAGE_MIN_SIZE = 384
IMAGE_MAX_SIZE = 640
LEARNING_RATE = 2e-4
BACKBONE_LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 0.1
AUX_WEIGHT = 0.5
FEATURE_LEVEL = 0
HORIZONTAL_FLIP_P = 0.5
USE_AMP = None  # CUDA에서는 자동 활성화, CPU에서는 자동 비활성화
DETERMINISTIC = True
SAVE_EPOCH_CHECKPOINTS = False  # True면 epoch별 checkpoint가 추가로 쌓임
RESUME_FROM = {}  # 예: {'shared_e2e': ROOT / 'cache/checkpoints/checkpoint_smoke_shared_e2e_seed42.pt'}

CONFIG = ExperimentConfig.for_run(
    ROOT, run_mode=RUN_MODE, seed=SEED, data_seed=DATA_SEED, experiments=EXPERIMENTS,
    train_images=TRAIN_IMAGES, val_images=VAL_IMAGES, epochs=EPOCHS,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    image_min_size=IMAGE_MIN_SIZE, image_max_size=IMAGE_MAX_SIZE,
    lr=LEARNING_RATE, backbone_lr=BACKBONE_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
    base_aux_weight=AUX_WEIGHT, feature_level=FEATURE_LEVEL,
    horizontal_flip_p=HORIZONTAL_FLIP_P, use_amp=USE_AMP,
    deterministic=DETERMINISTIC,
    save_epoch_checkpoints=SAVE_EPOCH_CHECKPOINTS,
)
CONFIG.as_dict()

{'root': 'D:\\gt-super',
 'run_mode': 'smoke',
 'checkpoint': 'SenseTime/deformable-detr',
 'train_images': 400,
 'val_images': 100,
 'epochs': 7,
 'batch_size': 2,
 'num_workers': 0,
 'image_size': {'shortest_edge': 384, 'longest_edge': 640},
 'lr': 0.0002,
 'backbone_lr': 2e-05,
 'weight_decay': 0.0001,
 'grad_clip': 0.1,
 'base_aux_weight': 0.5,
 'feature_level': 0,
 'horizontal_flip_p': 0.5,
 'use_amp': True,
 'deterministic': True,
 'save_epoch_checkpoints': False,
 'device': 'cuda',
 'experiments': ['baseline', 'shared_detach', 'shared_e2e'],
 'seed': 44,
 'data_seed': 42}

In [20]:
BUNDLE = prepare_data(CONFIG)
print({'train_images': len(BUNDLE.train_records), 'val_images': len(BUNDLE.val_records)})

VOC XML: 100%|██████████| 3750/3750 [00:01<00:00, 3228.42it/s]

Full split: train=3000 (9180 objects), val=750 (2530 objects)
Current run: train=400, val=100
{'train_images': 400, 'val_images': 100}


## 1. Baseline 학습


In [21]:
baseline_model, baseline_history, baseline_gradients = train_one_experiment(
    CONFIG, BUNDLE, experiment='baseline', seed=CONFIG.seed,
    resume_from=RESUME_FROM.get('baseline'),
)
baseline_final_map = float(baseline_history.iloc[-1]['map'])
baseline_model = release_model(baseline_model)
print({'experiment': 'baseline', 'seed': CONFIG.seed, 'final_mAP': baseline_final_map,
       'checkpoint': str(CONFIG.checkpoint_path('baseline'))})


===== baseline / seed=44 =====


Loading weights: 100%|██████████| 545/545 [00:00<00:00, 14898.72it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

[phase] initial main-only validation: 50 batches


[phase] initial validation complete: mAP=0.0060, AP@0.5=0.0151
[phase] training epoch 1/7: 200 batches


baseline e1:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


...

[phase] validating epoch 6/7: 50 batches


{'epoch': 6, 'main_loss': 1.5584, 'aux_loss': nan, 'aux_coverage': nan, 'collision_rate': nan, 'map': 0.0426, 'map50': 0.0818, 'map75': 0.0461}
[phase] training epoch 7/7: 200 batches


baseline e7:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[phase] validating epoch 7/7: 50 batches


{'epoch': 7, 'main_loss': 1.5141, 'aux_loss': nan, 'aux_coverage': nan, 'collision_rate': nan, 'map': 0.0434, 'map50': 0.0815, 'map75': 0.0467}
{'experiment': 'baseline', 'seed': 44, 'final_mAP': 0.04337302967905998, 'checkpoint': 'D:\\gt-super\\cache\\checkpoints\\checkpoint_smoke_baseline_seed44.pt'}


## 2. Shared-Detach 학습


In [22]:
detach_model, detach_history, detach_gradients = train_one_experiment(
    CONFIG, BUNDLE, experiment='shared_detach', seed=CONFIG.seed,
    resume_from=RESUME_FROM.get('shared_detach'),
)
detach_final_map = float(detach_history.iloc[-1]['map'])
detach_model = release_model(detach_model)
print({'experiment': 'shared_detach', 'seed': CONFIG.seed, 'final_mAP': detach_final_map,
       'checkpoint': str(CONFIG.checkpoint_path('shared_detach'))})


===== shared_detach / seed=44 =====


Loading weights: 100%|██████████| 545/545 [00:00<00:00, 15320.50it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

[phase] initial main-only validation: 50 batches


[phase] initial validation complete: mAP=0.0060, AP@0.5=0.0151
[phase] training epoch 1/7: 200 batches


shared_detach e1:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


...

[phase] validating epoch 6/7: 50 batches


{'epoch': 6, 'main_loss': 1.508, 'aux_loss': 1.4194, 'aux_coverage': 0.9992, 'collision_rate': 0.0008, 'map': 0.0571, 'map50': 0.1085, 'map75': 0.0568}
[phase] training epoch 7/7: 200 batches


shared_detach e7:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[phase] validating epoch 7/7: 50 batches


{'epoch': 7, 'main_loss': 1.444, 'aux_loss': 1.2959, 'aux_coverage': 0.9992, 'collision_rate': 0.0008, 'map': 0.0696, 'map50': 0.1188, 'map75': 0.0672}
{'experiment': 'shared_detach', 'seed': 44, 'final_mAP': 0.06955522298812866, 'checkpoint': 'D:\\gt-super\\cache\\checkpoints\\checkpoint_smoke_shared_detach_seed44.pt'}


## 3. Shared-E2E 학습


In [23]:
e2e_model, e2e_history, e2e_gradients = train_one_experiment(
    CONFIG, BUNDLE, experiment='shared_e2e', seed=CONFIG.seed,
    resume_from=RESUME_FROM.get('shared_e2e'),
)
e2e_final_map = float(e2e_history.iloc[-1]['map'])
e2e_model = release_model(e2e_model)
print({'experiment': 'shared_e2e', 'seed': CONFIG.seed, 'final_mAP': e2e_final_map,
       'checkpoint': str(CONFIG.checkpoint_path('shared_e2e'))})


===== shared_e2e / seed=44 =====


Loading weights: 100%|██████████| 545/545 [00:00<00:00, 15129.06it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

[phase] initial main-only validation: 50 batches


[phase] initial validation complete: mAP=0.0060, AP@0.5=0.0151
[phase] training epoch 1/7: 200 batches


shared_e2e e1:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


...

[phase] validating epoch 6/7: 50 batches


{'epoch': 6, 'main_loss': 1.4316, 'aux_loss': 0.9604, 'aux_coverage': 0.9992, 'collision_rate': 0.0008, 'map': 0.0588, 'map50': 0.0963, 'map75': 0.0595}
[phase] training epoch 7/7: 200 batches


shared_e2e e7:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[phase] validating epoch 7/7: 50 batches


{'epoch': 7, 'main_loss': 1.3825, 'aux_loss': 0.916, 'aux_coverage': 0.9992, 'collision_rate': 0.0008, 'map': 0.0635, 'map50': 0.1046, 'map75': 0.0636}
{'experiment': 'shared_e2e', 'seed': 44, 'final_mAP': 0.0634765550494194, 'checkpoint': 'D:\\gt-super\\cache\\checkpoints\\checkpoint_smoke_shared_e2e_seed44.pt'}
